# LAB 1 — PyTorch Tutorial: Building an Artificial Neuron

**Course:** Neural Networks (7th Semester)  
**Duration:** 3 Hours  
**Software:** Python, Jupyter Notebook / Google Colab, PyTorch, NumPy, Matplotlib

## Learning Objectives
- Understand the role of PyTorch in deep learning.
- Create and manipulate tensors.
- Perform tensor operations.
- Implement an artificial neuron from its mathematical equation.
- Apply activation functions using PyTorch.
- Understand automatic differentiation (Autograd).
- Create neurons using `nn.Linear`.

## Theory Primer: Concepts Behind This Lab

This lab is about the **single artificial neuron** — the smallest building block of every neural network, from a 2-layer perceptron to GPT-scale transformers. Before touching code, it helps to understand *why* each piece exists. This primer covers:
- Why tensors are used instead of plain Python lists / NumPy arrays
- Why the neuron equation is written as z = w·x + b
- Why activation functions (Sigmoid, Tanh, ReLU) are needed at all
- Why Autograd exists and how it fits into training
- Why nn.Linear is used instead of writing the neuron equation manually

### Why do we need tensors instead of plain Python lists/NumPy arrays?
A neural network is really just a chain of matrix multiplications and elementwise nonlinearities. We need a data structure that:
- Stores numbers in n-dimensional grids (scalars, vectors, matrices, batches of matrices, images, etc.) — this is a **tensor**.
- Can run the same operation on a CPU or a GPU without changing code (`.to('cuda')`).
- Can automatically remember *how* it was computed, so gradients can be derived later (autograd).

NumPy gives you (1), but not (2) or (3) built-in. That's the entire reason PyTorch tensors exist instead of just using NumPy arrays everywhere.

### 0.2 Why the artificial neuron looks like z = w·x + b
Biologically, a neuron receives multiple signals (dendrites), weighs their importance, sums them, and fires if the sum crosses a threshold. Mathematically we approximate this with:
- **Weights (w):** how much each input matters. Learned during training.
- **Bias (b):** shifts the decision boundary; lets the neuron fire even when all inputs are 0, or suppress firing even when inputs are large.
- **Dot product (w·x):** the efficient linear-algebra way to compute "multiply each input by its weight and add them up" in one operation instead of a manual loop.

This equation is *the same equation used inside every layer of every neural network* — a full layer is just many neurons (many w vectors) stacked into a weight matrix, and a full network is many layers of this chained together with nonlinearities in between.

### Why we need activation functions at all
If we only ever stacked `z = w·x + b` layers with no nonlinearity in between, the whole network would mathematically collapse into a single linear function no matter how many layers you add (a linear function of a linear function is still linear). Activation functions inject **nonlinearity**, which is what lets a network approximate curves, decision boundaries, and complex patterns instead of only straight lines/planes.

- **Sigmoid** `σ(z) = 1/(1+e^-z)`: squashes to (0,1). Historically popular for output layers doing binary probability, but saturates (near-zero gradient) for large |z|, which slows learning in deep nets.
- **Tanh** `tanh(z)`: squashes to (-1,1) and is zero-centered (unlike sigmoid), which tends to help gradient flow in hidden layers, but still saturates at the extremes.
- **ReLU** `max(0,z)`: the default choice in most modern hidden layers. It's cheap to compute and doesn't saturate for positive inputs, which is why it largely replaced sigmoid/tanh in hidden layers of deep networks — at the cost of a "dead neuron" problem when z stays negative.

### Why Autograd exists
Training a network means adjusting w and b to reduce a loss function. That requires the *gradient* of the loss with respect to every weight (via calculus/chain rule). Doing this by hand for a network with millions of parameters is infeasible. PyTorch's **Autograd** builds a computation graph as operations happen (because tensors have `requires_grad=True`) and then automatically applies the chain rule backward through that graph when you call `.backward()`. This is the mechanism that makes training possible at scale — we are only testing it on tiny scalar examples in this lab, but it's literally the same mechanism used to train billion-parameter models.

### Why nn.Linear instead of always writing `torch.dot(x, w) + b` manually
`nn.Linear(in_features, out_features)` does exactly the manual dot-product-plus-bias computation, but with three practical advantages:
1. **Automatic parameter management** — weights/bias are created for you (as `nn.Parameter` tensors) with `requires_grad=True` already set, so they're trainable out of the box.
2. **Sensible random initialization** — manually you'd have to pick initial values yourself; bad initialization can stall training. `nn.Linear` uses well-tested initialization schemes.
3. **Scales to layers, not just single neurons** — setting `out_features > 1` gives you many neurons (a full layer) in one line, and it composes cleanly inside `nn.Sequential`/custom `nn.Module` classes used for real networks.

In short: the manual version teaches you *what* the math is; `nn.Linear` is *how you'd actually build it* in a real project, and this lab proves they compute the same thing so you trust the abstraction.


## 1. Introduction to PyTorch

PyTorch is an open-source deep learning framework developed by Meta AI. It is widely used in academia and industry because of its **dynamic computation graph**, **Pythonic syntax**, and excellent debugging support. In deep learning, PyTorch lets us represent data as *tensors* (multi-dimensional arrays), build models out of layers, and automatically compute gradients for training — all of which we'll explore step by step in this lab.

## 2. First PyTorch Program

Before doing anything else, we check that PyTorch is installed correctly and confirm whether a CUDA-enabled GPU is available on this machine. `torch.__version__` displays the installed PyTorch version, and `torch.cuda.is_available()` returns `True`/`False` depending on GPU availability. We also import `torch.nn` (for building neural network layers later) and `matplotlib.pyplot` (for plotting activation functions further below).

In [ ]:
# %% [1] Importing PyTorch

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

**Output explanation:** **Output explanation:** The output confirms PyTorch 2.11.0 is installed and that CUDA is available (`True`), meaning a GPU can be used to accelerate computation if needed.

**Why we check this first:** Every subsequent line of code in this notebook depends on PyTorch being installed correctly, so this is a standard "sanity check" cell you'll see at the top of almost every PyTorch project/notebook. Checking `torch.cuda.is_available()` matters because tensors and models must explicitly be moved to a device (`.to('cuda')`) to use the GPU — PyTorch does NOT automatically use the GPU just because one exists. If this printed `False`, all our code would still run correctly, just slower (on CPU), since none of the tensors below are explicitly moved to GPU.

## 3. Creating Tensors

A **tensor** is PyTorch's fundamental data structure — a generalization of scalars, vectors, and matrices to any number of dimensions. Here we create tensors of different shapes and types using some of the most common constructors:
- `torch.tensor()` — builds a tensor directly from Python data (a 1D tensor here).
- `torch.zeros(shape)` — creates a tensor filled with zeros (a 2×3 matrix here).
- `torch.ones(shape)` — creates a tensor filled with ones (a 3×3 matrix here).
- `torch.rand(shape)` — creates a tensor with random values sampled uniformly from [0, 1) (a 3D tensor here).

In [ ]:
# %% [2] Creating Tensors of Different Shapes

a = torch.tensor([1, 2, 3])              # 1D tensor
b = torch.zeros((2, 3))                 # 2x3 tensor
c = torch.ones((3, 3))                  # 3x3 tensor
d = torch.rand((2, 2, 2))               # 3D tensor

print("\nTensors:")
print(a)
print(b)
print(c)
print(d)

**Output explanation:** **Output explanation:** `a` is a simple 1D tensor of integers. `b` is a 2×3 matrix of zeros, and `c` is a 3×3 matrix of ones — both default to `float32`. `d` is a 3D tensor (2×2×2) of random values between 0 and 1; since it's random, the exact numbers will differ each time this cell is run.

**Why we use these particular constructors (not much variation needed):** `torch.zeros` and `torch.ones` are used constantly in real networks to initialize things like bias vectors (start at 0) or masks (start at 1), and `torch.rand` is exactly the mechanism used internally to randomly initialize trainable weights (as we'll see later with `nn.Linear`). We deliberately create tensors of different ranks (1D, 2D, 3D) here because real data is rarely 1D: a batch of images is a 4D tensor (batch, channels, height, width), so getting comfortable with tensors of arbitrary shape early on avoids confusion later.

## 4. Tensor Properties

Every tensor carries metadata that describes it: `.shape` gives its dimensions, `.dtype` gives the data type of its elements (e.g. `int64`, `float32`), `.ndim` gives the number of dimensions (rank), and `.device` tells us whether the tensor currently lives on the CPU or GPU. Inspecting these properties is a useful debugging habit when building neural networks, since shape mismatches are one of the most common sources of errors.

In [ ]:
# %% [3] Tensor Properties

print(a.shape)
print(a.dtype)
print(a.ndim)
print(a.device)

**Output explanation:** **Output explanation:** `a.shape` is `torch.Size([3])` since `a` has 3 elements in one dimension. `a.dtype` is `int64` because `a` was created from Python integers. `a.ndim` is `1` (a 1D tensor), and `a.device` is `cpu` since the tensor lives in CPU memory (not on a GPU).

**Why we bother inspecting these four properties:** In practice, the single most common bug when building neural networks is a **shape mismatch** (e.g. trying to multiply a (3,) vector by a (5,) vector, or feeding a batch of the wrong size into a layer) — so `.shape` is the property you'll check constantly while debugging. `.dtype` matters because PyTorch is strict about type consistency: mixing `int64` and `float32` tensors in an operation often throws an error, so knowing `a` is `int64` here (not `float32` like most of our other tensors) explains why we'll almost always create tensors from floats (`2.` not `2`) elsewhere in this notebook. `.device` matters because an operation between a CPU tensor and a GPU tensor will error out — you must match devices explicitly.

## 5. Tensor Operations

PyTorch supports element-wise arithmetic (addition, subtraction, multiplication) directly through Python operators, as well as linear-algebra operations such as the **dot product** via `torch.dot()`. The dot product of two vectors `x` and `y` is computed as $\sum_i x_i y_i$, and it is the core mathematical operation underlying an artificial neuron, as we'll see in the next section.

In [ ]:
# %% [4] Arithmetic and Dot Product Operations

x = torch.tensor([2., 3., 4.])
y = torch.tensor([5., 6., 7.])

print("\nArithmetic Operations:")
print("Addition:", x + y)
print("Subtraction:", x - y)
print("Element-wise Multiplication:", x * y)
print("Dot Product:", torch.dot(x, y))

**Output explanation:** **Output explanation:** Addition, subtraction, and multiplication are performed element-wise between corresponding entries of `x` and `y`. The dot product sums the element-wise products: (2×5)+(3×6)+(4×7) = 10+18+28 = 56, matching the printed result.

**Why the dot product specifically matters here:** Elementwise +, -, * are used for things like combining feature maps or applying masks, but the **dot product** is singled out in this cell because it is literally the core operation of a neuron (Section 6). We introduce it here, isolated from the bias term, so you can verify you understand `torch.dot()` by hand before we add a bias and call it a "neuron." Nothing changes about how it's called — `torch.dot(x, y)` is the standard, idiomatic way to do this in PyTorch (as opposed to writing `sum(xi*yi for xi, yi in zip(x,y))`), because PyTorch's implementation runs as an optimized, vectorized C++/CUDA operation instead of a slow Python loop.

## 6. Artificial Neuron

An artificial neuron is the basic computational unit of a neural network. It takes a vector of inputs $x$, multiplies each input by a corresponding weight $w$, sums the results, and adds a bias term $b$:

$$z = w \cdot x + b$$

This is exactly the dot product operation we just practiced, plus a bias. Below, we implement this equation directly using `torch.dot()` for a neuron with 3 inputs.

In [ ]:
# %% [5] Artificial Neuron Equation z = w.x + b

x = torch.tensor([2., 3., 4.])
w = torch.tensor([0.5, 0.8, 0.2])
b = torch.tensor(1.)

z = torch.dot(x, w) + b
print("Weighted Sum =", z)

**Output explanation:** **Output explanation:** The weighted sum is computed as (2×0.5)+(3×0.8)+(4×0.2)+1 = 1+2.4+0.8+1 = 5.2, matching `Weighted Sum = tensor(5.2000)`. This value `z` will be used as the input to the activation functions in the next section.

**Why we compute z this way and not differently:** This single line (`torch.dot(x, w) + b`) is the direct, literal translation of the neuron equation z = w·x + b from Section 0.2/Section 6 into code — there's no "trick" here, it's intentionally the most explicit possible implementation so the connection between the math and the code is obvious. We keep `w` and `b` as separate, manually-chosen tensors (rather than randomly initialized, as `nn.Linear` will do later) specifically so you can hand-verify the arithmetic, which is why the explanation walks through the exact multiplication and addition above. Once we trust this by hand, Section 10 replaces manual `w`/`b` with `nn.Linear`'s auto-managed, trainable versions of the same numbers.

## 7. Activation Functions

The weighted sum `z` computed by a neuron is typically passed through a nonlinear **activation function** before being sent to the next layer. Nonlinearity is what allows neural networks to learn complex, non-linear patterns. The three most common activation functions are:
- **Sigmoid** — squashes values into the range (0, 1); useful for probabilities.
- **Tanh** — squashes values into the range (-1, 1); zero-centered.
- **ReLU** (Rectified Linear Unit) — outputs `max(0, z)`; simple and avoids vanishing gradients for positive inputs.

**Task:** Compare the outputs for different input values of `z`.

In [ ]:
# %% [6] Activation Functions

print(torch.sigmoid(z))
print(torch.tanh(z))
print(torch.relu(z))

**Output explanation:** **Output explanation:** With `z = 5.2`, Sigmoid squashes it close to 1 (≈0.9945) since large positive inputs push Sigmoid toward its upper bound. Tanh also saturates near its upper bound (≈0.9999) for the same reason. ReLU simply passes positive values through unchanged, so it outputs `5.2000` exactly.

**Why this single value (z=5.2) is a useful/illustrative test case:** Because z is a fairly large positive number, this example conveniently demonstrates the **saturation** behavior described in Section 0.3 — both Sigmoid and Tanh are already almost flat at their maximum here, meaning their gradient (slope) at this point is nearly zero. This is exactly the "vanishing gradient" problem that motivated the industry's shift toward ReLU: if this `z` were feeding into a deep network, a Sigmoid/Tanh neuron this saturated would barely update during training, while a ReLU neuron keeps a constant, healthy gradient of 1 for any positive input, which is why ReLU is preferred in hidden layers of most modern architectures.

## 8. Plot Activation Functions

To build intuition for how each activation function behaves, we plot Sigmoid, Tanh, and ReLU over a wide range of input values (from -10 to 10) using `torch.linspace()` and `matplotlib`. Notice how Sigmoid and Tanh saturate (flatten out) at the extremes, while ReLU stays linear for positive inputs and is exactly zero for negative inputs.

In [ ]:
# %% [7] Plot Sigmoid, Tanh, ReLU

x_vals = torch.linspace(-10, 10, 200)

plt.figure()
plt.plot(x_vals, torch.sigmoid(x_vals), label='Sigmoid')
plt.plot(x_vals, torch.tanh(x_vals), label='Tanh')
plt.plot(x_vals, torch.relu(x_vals), label='ReLU')
plt.title("Activation Functions")
plt.grid()
plt.legend()
plt.show()

**Output explanation:** **Output explanation:** The plot shows all three activation functions across inputs from -10 to 10. Sigmoid smoothly rises from 0 to 1, Tanh smoothly rises from -1 to 1 (both flattening/saturating at the extremes), and ReLU is flat at 0 for negative inputs and rises linearly (y = x) for positive inputs.

**Why we plot the whole range instead of just one point:** A single numeric output (like `z = 5.2` earlier) only shows behavior at one location; plotting from -10 to 10 lets you *see* the saturation regions (flat zones where the gradient ≈ 0) versus the sensitive regions (steep zones where a small input change causes a large output change). This visual is the standard way every deep learning course/textbook introduces activation functions, because it makes the abstract "vanishing gradient" argument from Section 0.3 concrete: you can literally see Sigmoid/Tanh going flat at the edges, while ReLU's positive half never flattens.

## 9. Automatic Differentiation (Autograd)

Training a neural network requires computing gradients of a loss function with respect to its parameters. PyTorch automates this through **Autograd**. When a tensor is created with `requires_grad=True`, PyTorch tracks every operation performed on it, building a computation graph behind the scenes. Calling `.backward()` on the final output then automatically computes $\frac{dy}{dx}$ and stores it in `x.grad`.

Here, for $y = x^2 + 3x + 1$, the analytical derivative is $\frac{dy}{dx} = 2x + 3$. At $x = 2$, this evaluates to $2(2) + 3 = 7$, which we should see printed below.

In [ ]:
# %% [8] Automatic Differentiation (Autograd)

x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1
y.backward()
print(x.grad)

**Output explanation:** **Output explanation:** The analytical derivative of y = x² + 3x + 1 is dy/dx = 2x + 3. At x = 2, this gives 2(2) + 3 = 7, which matches the printed gradient `tensor(7.)` — confirming Autograd computed it correctly.

**Why we cross-check against the hand-derived answer:** The whole point of this cell is to build trust in Autograd by verifying it against a case simple enough to differentiate by hand (basic calculus, power rule). In a real network you could never compute the gradient of the loss with respect to millions of weights by hand — so this toy example is a stand-in that lets you confirm PyTorch's automatic chain-rule machinery gives the mathematically correct answer, before trusting it on problems where you *can't* check the answer manually. `requires_grad=True` is what tells PyTorch to track `x` in the computation graph in the first place — without it, calling `.backward()` would raise an error, since PyTorch only builds a graph for tensors you've explicitly flagged as needing gradients.

## 10. Creating a Neuron Using `nn.Linear`

Rather than manually computing `w · x + b`, PyTorch provides the `nn.Linear` layer, which creates a neuron (or a whole layer of neurons) with **learnable** weights and bias, initialized randomly. `nn.Linear(in_features, out_features)` creates a layer that maps an input of size `in_features` to an output of size `out_features` — here, `nn.Linear(3, 1)` represents a single neuron with 3 inputs and 1 output, equivalent to the manual neuron we built earlier, except the weights and bias are now managed and trainable parameters.

In [ ]:
# %% [9] Creating a Neuron Using nn.Linear

layer = nn.Linear(3, 1)

print(layer.weight)
print(layer.bias)

inp = torch.tensor([[2., 3., 4.]])
print(layer(inp))

**Output explanation:** **Output explanation:** `nn.Linear(3, 1)` randomly initializes one weight per input (3 weights) and one bias. Because initialization is random, these exact numbers will differ on every run. The layer's output on `inp = [[2., 3., 4.]]` is the weighted sum of these random weights and bias, and `grad_fn=<AddmmBackward0>` shows PyTorch is already tracking this computation in the graph for potential backpropagation.

**Why we switch from manual weights to `nn.Linear` here:** This is the payoff of Section 0.5 — instead of us choosing `w` and `b` by hand (as in Section 6), `nn.Linear(3, 1)` creates them automatically as trainable parameters with `requires_grad=True` already set. Notice `inp` is written as `[[2., 3., 4.]]` (a 2D tensor of shape (1,3)) rather than a plain 1D vector — this is deliberate: PyTorch layers always expect a **batch dimension** first, even for a single example, because in real training you pass many examples through at once for efficiency. The `grad_fn=<AddmmBackward0>` in the output is proof that this layer is "wired into" Autograd already, unlike our earlier manual `torch.dot(x,w)+b`, which had no gradient tracking unless we set `requires_grad=True` ourselves.

---
# Student Tasks

The sections below complete the assigned student tasks:
1. Create tensors of different shapes.
2. Perform arithmetic and dot product operations.
3. Implement a neuron with 5 inputs.
4. Change the weights and bias; predict the output before execution.
5. Plot Sigmoid, Tanh and ReLU.
6. Use Autograd to compute the gradient of $y = x^2 + 5x + 2$.
7. Create `nn.Linear(5, 1)` and inspect its weights.
8. Compare the manual neuron implementation with `nn.Linear`.

## Task 3: Implement a Neuron with 5 Inputs

We now scale up the artificial neuron equation $z = w \cdot x + b$ to 5 inputs, using a 5-element input vector `x5`, a 5-element weight vector `w5`, and a scalar bias `b5`.

In [ ]:
# %% [4] Implement a Neuron with 5 Inputs

x5 = torch.tensor([1., 2., 3., 4., 5.])
w5 = torch.tensor([0.2, 0.4, 0.6, 0.8, 1.0])
b5 = torch.tensor(0.5)

z5 = torch.dot(x5, w5) + b5
print("\nNeuron Output (5 inputs):", z5)

**Output explanation:** **Output explanation:** Following z = w·x + b for 5 inputs: (1×0.2)+(2×0.4)+(3×0.6)+(4×0.8)+(5×1.0)+0.5 = 0.2+0.8+1.8+3.2+5.0+0.5 = 11.5, matching `tensor(11.5000)`.

**Why nothing about the code changes except the vector length:** This is intentional — it demonstrates that the neuron equation z = w·x + b (and `torch.dot()`) generalizes to *any* number of inputs without changing a single line of logic; only the length of `x` and `w` grows from 3 to 5. This is exactly why the dot product (rather than writing out `w1*x1 + w2*x2 + w3*x3...`) is the right abstraction: a real network layer might have hundreds of inputs, and the code stays identical either way.

## Task 4: Change Weights & Bias (Predict Before Running)

Here we change the weights to all 1's and the bias to 0, then recompute the neuron's output. Before running this cell, try to predict the result manually: with all weights equal to 1, the weighted sum simply becomes the sum of the inputs (1+2+3+4+5 = 15), plus a bias of 0, giving an expected output of 15.

In [ ]:
# %% [5] Change Weights & Bias (Predict before running)

# Try predicting manually before execution
w5_new = torch.tensor([1., 1., 1., 1., 1.])
b5_new = torch.tensor(0.)

z5_new = torch.dot(x5, w5_new) + b5_new
print("New Output (all weights=1, bias=0):", z5_new)

**Output explanation:** **Output explanation:** As predicted, with all weights set to 1 and bias set to 0, the neuron output simply becomes the sum of the inputs: 1+2+3+4+5 = 15, matching `tensor(15.)` exactly.

**Why this "predict-before-running" exercise is useful:** Setting weights to all-1s and bias to 0 is a deliberately simple edge case that strips the neuron equation down to plain summation, making it easy to verify your understanding without a calculator. More importantly, it demonstrates *why weights and bias matter*: with different (e.g. random or learned) weights, the same inputs `x5` would produce a completely different output — the weights control how much each input contributes, and the bias shifts the result up or down regardless of the inputs. This is the intuition behind what "training a neuron" actually optimizes: search for the w and b values that make the neuron's output match what you want it to predict.

## Task 5: Plot Sigmoid, Tanh, ReLU

As in Section 8 above, we plot the three activation functions together over the range [-10, 10] to visually compare their shapes and saturation behavior.

In [ ]:
# %% [6] Plot Sigmoid, Tanh, ReLU

x_vals = torch.linspace(-10, 10, 200)

plt.figure()
plt.plot(x_vals, torch.sigmoid(x_vals), label='Sigmoid')
plt.plot(x_vals, torch.tanh(x_vals), label='Tanh')
plt.plot(x_vals, torch.relu(x_vals), label='ReLU')
plt.title("Activation Functions")
plt.grid()
plt.legend()
plt.show()

**Output explanation:** Same plot as Section 8 — Sigmoid and Tanh both show their characteristic S-curve saturation, while ReLU stays at 0 for negative values and increases linearly for positive values.

## Task 6: Autograd — Gradient of $y = x^2 + 5x + 2$

We use Autograd to compute the derivative of $y = x^2 + 5x + 2$ at $x = 2$. Analytically, $\frac{dy}{dx} = 2x + 5$, so at $x=2$ the expected gradient is $2(2) + 5 = 9$. PyTorch computes this automatically via `y.backward()`, storing the result in `x.grad`.

In [ ]:
# %% [7] Autograd: Gradient of y = x^2 + 5x + 2

x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 5*x + 2

y.backward()

print("\nGradient dy/dx at x=2:", x.grad)   # Expected: 2x + 5 = 9

**Output explanation:** **Output explanation:** For y = x² + 5x + 2, the derivative is dy/dx = 2x + 5. At x = 2, this is 2(2) + 5 = 9, matching the printed gradient `tensor(9.)`.

**Why we repeat the autograd exercise with different coefficients:** The code is intentionally almost identical to Section 9 — same structure (`requires_grad=True` → define `y` → `.backward()` → read `.grad`) but with different coefficients (5 and 2 instead of 3 and 1). This repetition is deliberate: it proves Autograd isn't just "memorizing" the first example — it recomputes the correct symbolic derivative for whatever expression you give it, which is exactly the general-purpose behavior needed to differentiate the loss function of a real (much more complex) neural network automatically.

## Task 7: `nn.Linear(5, 1)` and Inspecting Weights

We now create an `nn.Linear` layer with 5 inputs and 1 output, matching the 5-input neuron from Task 3. We inspect its randomly initialized `weight` and `bias` parameters, then feed it a sample input to see the computed output.

In [ ]:
# %% [8] nn.Linear(5,1) and Inspect Weights

layer = nn.Linear(5, 1)

print("\nLinear Layer Weights:", layer.weight)
print("Linear Layer Bias:", layer.bias)

inp = torch.tensor([[1., 2., 3., 4., 5.]])
output = layer(inp)

print("Layer Output:", output)

**Output explanation:** **Output explanation:** `nn.Linear(5, 1)` randomly initializes 5 weights and 1 bias (values will differ each run). Feeding in `[[1., 2., 3., 4., 5.]]` produces the weighted sum of these random parameters — here `2.8008` — which will be verified against a manual calculation in the next task.

**Why we mirror Task 3's 5-input neuron here:** This cell repeats Section 10's `nn.Linear` pattern, but now matching the 5-input dimensionality of Task 3, to directly set up the comparison in Task 8. The code doesn't change conceptually — only `in_features` changes from 3 to 5 — reinforcing that `nn.Linear(in_features, out_features)` scales to any input size with no other code changes, exactly like the manual dot product did in Task 3.

## Task 8: Compare Manual Neuron vs `nn.Linear`

Finally, we verify that `nn.Linear` is doing exactly the same computation as our manual neuron equation. We extract the weight and bias that `nn.Linear` initialized internally, manually compute `w · x + b` using `torch.dot()`, and compare it against the layer's own output. The difference between the two should be (numerically) zero, confirming that `nn.Linear` is just a convenient, trainable wrapper around the same underlying math.

In [ ]:
# %% [9] Compare Manual Neuron vs nn.Linear

# Manual computation using same weights
manual_w = layer.weight.data[0]   # extract weights
manual_b = layer.bias.data

manual_output = torch.dot(inp[0], manual_w) + manual_b

print("\nManual Output:", manual_output)
print("nn.Linear Output:", output)

print("Difference:", manual_output - output)

**Output explanation:** **Output explanation:** The manual computation using `torch.dot()` on the layer's own weights and bias produces the same value (`2.8008`) as `layer(inp)`. The tiny difference (`-2.3842e-07`) is not a real discrepancy — it's floating-point rounding error, confirming that `nn.Linear` internally performs exactly the same `w·x + b` computation as our manual neuron.

**Why this final comparison matters (ties the whole lab together):** This is the "proof" cell for everything the theory in Section 0.5 claimed: that `nn.Linear` is not some different, mysterious computation — it is exactly `torch.dot(x, w) + b` under the hood, just with auto-managed, trainable parameters and (internally) a slightly different but numerically equivalent implementation (`addmm`, matrix-multiply-add, which is why the difference is ~1e-7 float rounding error rather than exactly 0). Understanding this equivalence is what lets you confidently use `nn.Linear` (and later, stacks of them) in real models, trusting that it's doing the same fundamental neuron math you verified by hand throughout this lab, just packaged for training at scale.